Github Analysis!

In [ ]:
# Use the correct function name from your file
from spark_utils import load_github_bronze_keyword_extractions_data
spark, df = load_github_bronze_keyword_extractions_data()

In [ ]:
# Simpler version without imports
df = spark.read.format("delta").load("s3a://delta-lake/bronze/github/keyword_extractions")

print("=== TOP KEYWORDS (Simple) ===")
df.groupBy("keyword") \
  .sum("mentions") \
  .orderBy("sum(mentions)", ascending=False) \
  .limit(10) \
  .show(truncate=False)

print("\n=== BASIC STATS ===")
df.describe("mentions").show()
df.select("keyword").distinct().count()

In [ ]:
# Load your data
df = spark.read.format("delta").load("s3a://delta-lake/bronze/github/keyword_extractions")

print("=== AI/ML TRENDS IN GITHUB ACTIVITY ===")
# Filter for AI-related keywords from your taxonomy
ai_keywords = ['gpt-4o', 'claude', 'llama', 'pytorch', 'tensorflow', 'transformers', 'langchain']

ai_df = df.filter(df.keyword.isin(ai_keywords))
ai_df.groupBy("keyword") \
  .sum("mentions") \
  .orderBy("sum(mentions)", ascending=False) \
  .show()

print("\n=== BLOCKCHAIN ACTIVITY ===")
blockchain_keywords = ['ethereum', 'solana', 'near', 'web3', 'solidity']
blockchain_df = df.filter(df.keyword.isin(blockchain_keywords))
blockchain_df.groupBy("keyword") \
  .sum("mentions") \
  .orderBy("sum(mentions)", ascending=False) \
  .show()

print("\n=== DEVOPS TRENDS ===")
devops_keywords = ['docker', 'kubernetes', 'terraform', 'github actions', 'jenkins']
devops_df = df.filter(df.keyword.isin(devops_keywords))
devops_df.groupBy("keyword") \
  .sum("mentions") \
  .orderBy("sum(mentions)", ascending=False) \
  .show()

print("\n=== HOURLY DEVELOPER ACTIVITY PATTERN ===")
df.groupBy("hour") \
  .sum("mentions") \
  .orderBy("hour") \
  .show()

In [ ]:
# Let's see the complete hourly pattern and analyze peak times
print("=== COMPLETE HOURLY PATTERN ===")
hourly_pattern = df.groupBy("hour") \
  .sum("mentions") \
  .orderBy("hour") \
  .collect()

for row in hourly_pattern:
    hour = row['hour']
    mentions = row['sum(mentions)']
    # Convert to visual bar
    bar = "█" * max(1, int(mentions / 2000))
    print(f"Hour {hour:2d}: {mentions:6,} {bar}")

print("\n=== NEAR vs ETHEREUM DEEP DIVE ===")
# Why is NEAR so close to Ethereum?
near_vs_eth = df.filter(df.keyword.isin(['near', 'ethereum']))
near_vs_eth.groupBy("keyword", "hour") \
  .sum("mentions") \
  .orderBy("keyword", "hour") \
  .show(50)